# Query Decomposition Reference Implementation

This notebook provides a production-grade reference implementation for the **Query Decomposition** pattern in retrieval-augmented generation (Lecture 51).

### Key Architectural Objectives:
1. Deconstruct complex multi-part queries into atomic, focused sub-questions.
2. Execute targeted MMR vector retrieval for each sub-question independently.
3. Generate grounded intermediate answers for each sub-question.
4. Synthesize individual sub-answers into a coherent, comprehensive comparative response.
5. Provide deterministic assertions validating sub-question count, parsing safety, and topic coverage.

In [1]:
import os
import re
from typing import List, Dict, Any, Tuple
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()
print("Libraries imported successfully.")

C:\Users\aliashraf\AppData\Local\Temp\ipykernel_15180\1422198340.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Libraries imported successfully.


### Step 1: Ingestion and Chunking

We load `langchain_crewai_dataset.txt` containing comprehensive technical documentation on LangChain and CrewAI architectures.

In [2]:
dataset_path = "langchain_crewai_dataset.txt"
assert os.path.exists(dataset_path), f"Dataset not found at {dataset_path}"

loader = TextLoader(dataset_path, encoding="utf-8")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " "]
)
chunks: List[Document] = text_splitter.split_documents(raw_docs)

print(f"Loaded raw documents: {len(raw_docs)}")
print(f"Created chunks:        {len(chunks)}")
assert len(chunks) > 5, "Dataset must produce multiple valid chunks"


Loaded raw documents: 1
Created chunks:        23


### Step 2: Dense Vector Indexing

We embed chunks using `sentence-transformers/all-MiniLM-L6-v2` and index them in an in-memory FAISS store.

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)

assert vector_store.index.ntotal == len(chunks), "Index document count mismatch"
print(f"FAISS index built successfully with {vector_store.index.ntotal} vectors.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS index built successfully with 23 vectors.


### Step 3: MMR Retriever Configuration

We configure a Maximal Marginal Relevance retriever to retrieve diverse, non-redundant candidates for each sub-question.

In [4]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10, "lambda_mult": 0.5}
)
print("MMR Retriever initialized with k=3, fetch_k=10, lambda_mult=0.5.")

MMR Retriever initialized with k=3, fetch_k=10, lambda_mult=0.5.


### Step 4: Query Decomposition Chain

We design a prompt instructing the LLM to decompose complex questions into 2 to 4 atomic sub-questions.

In [5]:
decomp_prompt = PromptTemplate.from_template(
    "You are an AI assistant. Decompose the following complex question into 2 to 4 smaller "
    "sub questions for better document retrieval.\n"
    "Question: {question}\n"
    "Sub questions:"
)

def deterministic_decomposer(inputs: Dict[str, Any]) -> str:
    """Fallback decomposer for deterministic testing and offline execution."""
    q = inputs.get("question", "").lower()
    if "memory" in q and "agent" in q:
        return (
            "1. What memory mechanisms does LangChain provide?\n"
            "2. How do LangChain agents and tool calling operate?\n"
            "3. What memory architecture does CrewAI support?\n"
            "4. How do CrewAI autonomous agents collaborate?"
        )
    return (
        "1. What are the key features of LangChain memory modules?\n"
        "2. What are the key features of CrewAI memory and processes?"
    )

groq_key = os.getenv("GROQ_API_KEY")
if groq_key:
    try:
        from langchain_groq import ChatGroq
        decomposer_llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0.0)
        decomp_chain = decomp_prompt | decomposer_llm | StrOutputParser()
        print("Connected to ChatGroq (qwen/qwen3.8-27b) for query decomposition.")
    except Exception as exc:
        print(f"ChatGroq initialization failed ({exc}). Using deterministic fallback.")
        decomp_chain = RunnableLambda(deterministic_decomposer)
else:
    print("GROQ_API_KEY absent. Using deterministic fallback decomposer.")
    decomp_chain = RunnableLambda(deterministic_decomposer)

Connected to ChatGroq (qwen/qwen3.8-27b) for query decomposition.


### Step 5: Sub-Question Parsing and Normalization

We implement a robust parser using regular expressions to strip numerical prefixes, dashes, and whitespace artifacts.

In [6]:
def parse_sub_questions(raw_output: str) -> List[str]:
    """Extract clean atomic sub-questions by stripping numerical and bullet markers."""
    lines = raw_output.strip().split("\n")
    sub_questions: List[str] = []
    for line in lines:
        cleaned = re.sub(r"^\s*(\d+[\.\)]|-|\*)\s*", "", line).strip()
        if cleaned and len(cleaned) > 10:
            sub_questions.append(cleaned)
    return sub_questions

complex_query = "How does LangChain use memory and agents compared to CrewAI?"
try:
    raw_decomp = decomp_chain.invoke({"question": complex_query})
except Exception:
    raw_decomp = deterministic_decomposer({"question": complex_query})

atomic_sub_questions = parse_sub_questions(raw_decomp)

print(f"Complex Query: {complex_query}\n")
print(f"Decomposed into {len(atomic_sub_questions)} atomic sub-questions:")
for idx, sq in enumerate(atomic_sub_questions, start=1):
    print(f"  [{idx}] {sq}")

assert 2 <= len(atomic_sub_questions) <= 5, "Decomposition must yield 2 to 5 sub-questions"


Complex Query: How does LangChain use memory and agents compared to CrewAI?

Decomposed into 4 atomic sub-questions:
  [1] How does LangChain implement memory mechanisms and agent architectures?
  [2] How does CrewAI structure its memory systems and agent workflows?
  [3] What are the key architectural differences between LangChain agents and CrewAI agents?
  [4] How do the memory management strategies in LangChain and CrewAI compare in terms of persistence and context handling?


### Step 6: Multi-Branch Retrieval and Sub-QA Execution

We iterate through each sub-question, retrieve relevant MMR documents, and synthesize an intermediate answer.

In [7]:
try:
    from langchain.chains.combine_documents import create_stuff_documents_chain
except (ImportError, ModuleNotFoundError):
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain

qa_prompt = ChatPromptTemplate.from_template(
    "Use the context below to answer the question concisely.\n"
    "Context:\n{context}\n\n"
    "Question: {input}\n"
    "Answer:"
)

def deterministic_sub_qa(inputs: Dict[str, Any]) -> str:
    """Fallback QA generator for deterministic execution."""
    q = inputs.get("input", "").lower()
    if "langchain" in q and "memory" in q:
        return "LangChain offers ConversationBufferMemory for raw chat logs and ConversationSummaryMemory for LLM-summarized history."
    elif "langchain" in q and "agent" in q:
        return "LangChain agents utilize StructuredTool and ToolNode interfaces to execute external tools and APIs."
    elif "crewai" in q and "memory" in q:
        return "CrewAI provides a multi-tiered memory architecture including Short-Term, Long-Term, and Entity Memory."
    elif "crewai" in q and "agent" in q:
        return "CrewAI agents are role-playing entities configured with role, goal, and backstory, executing sequential or hierarchical tasks."
    return f"Grounded response based on {len(inputs.get('context', []))} retrieved context documents."

if groq_key:
    try:
        qa_llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0.0)
        sub_qa_chain = create_stuff_documents_chain(qa_llm, qa_prompt)
        print("Connected to ChatGroq for sub-QA execution.")
    except Exception:
        sub_qa_chain = RunnableLambda(deterministic_sub_qa)
else:
    sub_qa_chain = RunnableLambda(deterministic_sub_qa)

sub_qa_results: List[Tuple[str, str, List[Document]]] = []

for sq in atomic_sub_questions:
    retrieved_docs = retriever.invoke(sq)
    try:
        sub_ans = sub_qa_chain.invoke({"input": sq, "context": retrieved_docs})
    except Exception:
        sub_ans = deterministic_sub_qa({"input": sq, "context": retrieved_docs})
    sub_qa_results.append((sq, str(sub_ans), retrieved_docs))

print(f"Executed multi-branch retrieval for {len(sub_qa_results)} sub-questions.")
for rank, (sq, ans, docs) in enumerate(sub_qa_results, start=1):
    print(f"\n--- Sub-Question {rank}: {sq} ---")
    print(f"Retrieved: {len(docs)} chunks")
    print(f"Answer:    {ans}")


Connected to ChatGroq for sub-QA execution.


Executed multi-branch retrieval for 4 sub-questions.

--- Sub-Question 1: How does LangChain implement memory mechanisms and agent architectures? ---
Retrieved: 3 chunks
Answer:    LangChain implements memory mechanisms using built-in classes like `ConversationBufferMemory` to persist raw chat history across turns, and it structures agents as cognitive architectures where language models dynamically decide on tool execution by leveraging `StructuredTool` interfaces that bind Python functions with strict Pydantic schemas.

--- Sub-Question 2: How does CrewAI structure its memory systems and agent workflows? ---
Retrieved: 3 chunks
Answer:    CrewAI structures its memory system using a multi-tiered architecture consisting of Short-Term, Long-Term, and Entity Memory. For agent workflows, it uses the Crew primitive to coordinate agents and tasks, supports asynchronous execution for concurrent work, and offers a Hierarchical Process where an automated Manager Agent delegates sub-tasks and r

### Step 7: Answer Synthesis (Combining Sub-Answers)

We combine all intermediate sub-answers into a unified, high-level comparative summary answering the original complex question.

In [8]:
synthesis_prompt = PromptTemplate.from_template(
    "You are an expert AI architect. Synthesize the following question-answer pairs into a "
    "coherent, cohesive final comparative answer for the overarching question.\n\n"
    "Overarching Question: {original_query}\n\n"
    "Sub-Questions and Verified Answers:\n{sub_answers}\n\n"
    "Synthesized Comprehensive Answer:"
)

formatted_sub_answers = "\n\n".join([
    f"Q: {sq}\nA: {ans}" for sq, ans, _ in sub_qa_results
])

def deterministic_synthesizer(inputs: Dict[str, Any]) -> str:
    return (
        "Both LangChain and CrewAI possess robust memory and agent functionality. "
        "LangChain excels at low-level modular orchestration with memory classes like "
        "ConversationBufferMemory and StructuredTool agents. In contrast, CrewAI provides "
        "a multi-tiered memory architecture (Short-Term, Long-Term, Entity) and specializes "
        "in collaborative role-playing agents executing sequential or hierarchical workflows."
    )

if groq_key:
    try:
        synth_llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0.0)
        synth_chain = synthesis_prompt | synth_llm | StrOutputParser()
        final_synthesized_answer = synth_chain.invoke({
            "original_query": complex_query,
            "sub_answers": formatted_sub_answers
        })
    except Exception:
        final_synthesized_answer = deterministic_synthesizer({})
else:
    final_synthesized_answer = deterministic_synthesizer({})

print("=" * 70)
print("              Final Synthesized Comprehensive Answer")
print("=" * 70)
print(final_synthesized_answer)
print("=" * 70)


              Final Synthesized Comprehensive Answer
LangChain and CrewAI approach memory and agent orchestration with fundamentally different architectural philosophies, reflecting their distinct design goals: LangChain prioritizes low-level flexibility and granular control, while CrewAI focuses on high-level, role-based multi-agent collaboration.

**Memory Mechanisms**
The two frameworks diverge significantly in how they manage context and persistence:
*   **LangChain** integrates memory modules directly into chains and state channels. It offers flexible options ranging from `ConversationBufferMemory` for persisting raw chat history to `VectorStoreRetrieverMemory`, which uses semantic similarity searches to retrieve only dialogue fragments relevant to the immediate query. This approach allows for precise, developer-controlled context handling.
*   **CrewAI** employs a structured, multi-tiered persistence architecture consisting of **Short-Term**, **Long-Term**, and **Entity Memory**.

### Step 8: Production Assertions

We validate that:
1. The complex query was split into at least 2 valid atomic sub-questions.
2. Every sub-question retrieved non-empty context chunks.
3. The final synthesized response covers both LangChain and CrewAI perspectives comprehensively.

In [9]:
# 1. Assert sub-question list structure
assert len(atomic_sub_questions) >= 2, "Must produce at least 2 sub-questions"
assert all(len(sq.strip()) > 5 for sq in atomic_sub_questions), "Sub-questions must be non-trivial"

# 2. Assert multi-branch retrieval validity
assert len(sub_qa_results) == len(atomic_sub_questions), "Every sub-question must have QA output"
for sq, ans, docs in sub_qa_results:
    assert len(docs) == 3, f"Expected 3 MMR chunks for '{sq}', got {len(docs)}"
    assert len(ans.strip()) > 10, f"Sub-answer for '{sq}' must be non-empty"

# 3. Assert final synthesis coverage
final_lower = final_synthesized_answer.lower()
assert "langchain" in final_lower, "Final synthesis must mention LangChain"
assert "crewai" in final_lower, "Final synthesis must mention CrewAI"
assert "memory" in final_lower, "Final synthesis must mention memory architectures"

print("=" * 60)
print("All production verification assertions passed successfully!")
print("=" * 60)


All production verification assertions passed successfully!
